# Портрет гостей сети

Работа по обновлённому шаблону главы 3. В публичном отчёте только агрегаты; частные файлы сохраняются в системном временном каталоге.

In [1]:
import sys
from pathlib import Path

import pandas as pd

def data_folder(*files):
    for folder in (Path("data"), Path("../data")):
        if all((folder / filename).is_file() for filename in files):
            print("Папка данных:", folder.resolve())
            return folder
    raise FileNotFoundError("В data/ и ../data/ нет полного набора: " + ", ".join(files))

print("Python", sys.version.split()[0], "| pandas", pd.__version__)
CLIENTS_FILE = "clients_selected.csv"
DATA = data_folder(CLIENTS_FILE, "orders_3m.csv", "shops.csv")
import json
import tempfile
import shutil
from IPython.display import display, Markdown

Python 3.12.14 | pandas 3.0.6
Папка данных: /private/tmp/kuch03round/student-retest/data


## 1. Цель
Описать участников программы лояльности и их осенние покупки. Заказчик — маркетинг. Результат — агрегаты по городу, возрасту и источнику. Успех — повторяемый отчёт, сохранение выручки и проверенные ключи.

## 2. Данные
Анкеты, заказы за сентябрь–ноябрь 2026 и справочник кофеен. Выбор выгрузки в CLIENTS_FILE первой ячейки.

In [2]:
raw = pd.read_csv(DATA / CLIENTS_FILE)
orders = pd.read_csv(DATA / 'orders_3m.csv', parse_dates=['datetime'])
shops = pd.read_csv(DATA / 'shops.csv')
# Частный осмотр: строки сохранены в памяти, в публичном выводе только размеры.

print('Размеры:',raw.shape, orders.shape, shops.shape)
print(pd.DataFrame({'тип':raw.dtypes.astype(str),'пропуски':raw.isna().sum(),'различных':raw.nunique()}))
print('Полных дублей:',raw.duplicated().sum())
print('Написаний города:',raw.city.nunique())

Размеры: (309, 9) (2089, 9) (7, 5)
             тип  пропуски  различных
client_id    str         0        300
name         str         0         34
gender       str        19          6
birth_date   str        19        249
city         str        16         29
phone        str        18        277
signup_date  str         0        266
bonus        str        15        138
source       str        27         14
Полных дублей: 9
Написаний города: 29


**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** 307 анкет, 2089 заказов. 8 полных дублей; 29 написаний города. Бонусы текстовые, даты смешаны, простого удаления пустых строк недостаточно.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 3. Предобработка
Полные копии удаляются до нормализации ключа, повторные анкеты — по свежей дате. Частный файл находится в закрытой папке private/ внутри временного каталога.

In [3]:

SYNONYMS = {"соц. сети": "соцсети", "от друзей": "друзья", "проходил мимо": "мимо",
            "2гис": "карты", "яндекс карты": "карты"}

def clean_city(s):
    return (s.str.strip().str.lower().str.removeprefix("г.").str.strip()
             .replace({"екб": "екатеринбург"}).str.capitalize())

def clean_phone(s):
    digits = s.str.replace(r"\D", "", regex=True)
    ok = digits.str.fullmatch(r"[78]?\d{10}")      # 10 цифр или 11 с кодом 7/8
    return digits.where(ok).str[-10:]               # другие длины — пропуск, а не «похожий» номер

def clean_gender(s):
    return s.str.strip().str.lower().map({"м": "М", "муж": "М", "ж": "Ж", "жен": "Ж"})

def parse_birth(s):
    iso = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    ru = pd.to_datetime(s, format="%d.%m.%Y", errors="coerce")
    born = iso.fillna(ru)
    return born.where((2026 - born.dt.year - (born.dt.strftime('%m%d') > '1201')).between(14, 100))

def clean_bonus(s):
    b = pd.to_numeric(s.str.replace(" ", ""), errors="coerce")
    return b.mask((b < 0) | (b > 10_000))

def clean_clients(raw):
    df = raw.drop_duplicates().copy()
    df["client_id"] = df.client_id.str.strip().str.upper()
    df["name"] = df.name.str.strip().str.capitalize()
    for col, fun in {"gender":clean_gender,"birth_date":parse_birth,"city":clean_city,"phone":clean_phone,"bonus":clean_bonus}.items():
        df[col] = fun(df[col])
    df["signup_date"] = pd.to_datetime(df.signup_date)
    df["source"] = df.source.str.strip().str.lower().replace(SYNONYMS)
    return df.sort_values("signup_date").drop_duplicates("client_id",keep="last").reset_index(drop=True)


raw_snapshot = raw.copy(deep=True)
clients = clean_clients(raw)
log = {'выгрузка':len(raw),'без полных дублей':len(raw.drop_duplicates()),'анкет':len(clients)}
assert clients.client_id.notna().all() and clients.client_id.is_unique
assert clients.gender.dropna().isin(['М','Ж']).all()
assert clients.city.dropna().isin(['Пермь','Казань','Тюмень','Уфа','Екатеринбург','Москва']).all()
assert clients.bonus.dropna().between(0,10000).all()
assert raw.equals(raw_snapshot)
PRIVATE = Path(tempfile.mkdtemp(prefix='crm-private-')) / 'private'
PRIVATE.mkdir()
clients.to_csv(PRIVATE / 'clients_clean.csv',index=False)
n_exact = log['выгрузка']-log['без полных дублей']
n_versions = log['без полных дублей']-log['анкет']
assert log['выгрузка']-n_exact-n_versions == log['анкет']
print('Выбранная выгрузка:',CLIENTS_FILE)
print(log)
print('Удалено полных копий:',n_exact,'; повторных анкет:',n_versions)
display(Markdown(f"**Текущий журнал ({CLIENTS_FILE}):** вход {len(raw)}, полных копий удалено {n_exact}, повторных анкет {n_versions}, осталось {len(clients)}. Контроль: {len(raw)} − {n_exact} − {n_versions} = {len(clients)}."))

Выбранная выгрузка: clients_selected.csv
{'выгрузка': 309, 'без полных дублей': 300, 'анкет': 294}
Удалено полных копий: 9 ; повторных анкет: 6


**Текущий журнал (clients_selected.csv):** вход 309, полных копий удалено 9, повторных анкет 6, осталось 294. Контроль: 309 − 9 − 6 = 294.

**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** Для исходной выгрузки 307 → 299 → 293: восемь полных копий и шесть повторных анкет. Числа повторно вычисляются по выбранному CLIENTS_FILE; новая полная копия должна увеличить только загрузку и удалённые копии.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 3.1. Качество данных
Поиск проблем воспроизводим по полю и правилу. Публичный отчёт не содержит примеров карт; внутренние примеры сохраняются отдельно.

In [4]:
missing_pct = (clients.isna().mean()*100).round(1).sort_values(ascending=False)
usable_age = round(clients.birth_date.notna().mean()*100,1)
print(missing_pct)
born = pd.to_datetime(raw.birth_date,format='%Y-%m-%d',errors='coerce').fillna(pd.to_datetime(raw.birth_date,format='%d.%m.%Y',errors='coerce'))
bonus_num = pd.to_numeric(raw.bonus.str.replace(' ',''),errors='coerce')
tickets = []
for field,mask,rule,query in [('birth_date',born.eq('1900-01-01'),'дата рождения не заглушка',"birth_date = '1900-01-01'"),('bonus',bonus_num.lt(0),'баланс не меньше 0','bonus < 0'),('bonus',bonus_num.gt(10000),'баланс не выше 10000','bonus > 10000')]:
    tickets.append({'где':'clients_raw.csv '+field,'правило':rule,'строк':int(mask.sum()),'всего':len(raw),'как_найти':query,'ожидание':rule,'влияние':'искажает профиль и бонусные рассылки','обход':'поле заменено пропуском'})
internal_tickets = []
for field,mask in [('birth_date',born.eq('1900-01-01')),('bonus',bonus_num.lt(0)),('bonus',bonus_num.gt(10000))]:
    internal_tickets.append({'поле':field,'примеры':raw.loc[mask,'client_id'].head(5).tolist()})
(PRIVATE / 'tickets.json').write_text(json.dumps(internal_tickets,ensure_ascii=False))
print(tickets)
print('Возраст известен, %:',usable_age)

birth_date     14.3
source          9.2
bonus           7.8
gender          6.1
phone           5.8
city            5.4
client_id       0.0
name            0.0
signup_date     0.0
dtype: float64
[{'где': 'clients_raw.csv birth_date', 'правило': 'дата рождения не заглушка', 'строк': 3, 'всего': 309, 'как_найти': "birth_date = '1900-01-01'", 'ожидание': 'дата рождения не заглушка', 'влияние': 'искажает профиль и бонусные рассылки', 'обход': 'поле заменено пропуском'}, {'где': 'clients_raw.csv bonus', 'правило': 'баланс не меньше 0', 'строк': 2, 'всего': 309, 'как_найти': 'bonus < 0', 'ожидание': 'баланс не меньше 0', 'влияние': 'искажает профиль и бонусные рассылки', 'обход': 'поле заменено пропуском'}, {'где': 'clients_raw.csv bonus', 'правило': 'баланс не выше 10000', 'строк': 1, 'всего': 309, 'как_найти': 'bonus > 10000', 'ожидание': 'баланс не выше 10000', 'влияние': 'искажает профиль и бонусные рассылки', 'обход': 'поле заменено пропуском'}]
Возраст известен, %: 85.7


**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** Возраст известен у 85.7% анкет; 14.3% дат неизвестны. Удаление аномалии из поля сохраняет полезную анкету. Средний возраст описывает только известные даты; MNAR возможен, представительность неизвестна. Тикеты воспроизводимы по условию; номера строк служат только для этой выгрузки.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 4.1. Возраст и города

In [5]:
clients["age"] = 2026-clients.birth_date.dt.year-(clients.birth_date.dt.strftime("%m%d")>"1201")
clients["age_group"] = pd.cut(clients.age,bins=[14,25,35,45,60,101],labels=["14–24","25–34","35–44","45–59","60–100"],right=False)
median_age = clients.age.median()
age_counts = clients.age_group.value_counts().to_dict()
by_city = clients.city.value_counts().to_dict()
print('Медиана возраста:',median_age)
print(age_counts)
print(by_city)
display(Markdown(f"**Текущий портрет ({CLIENTS_FILE}):** {len(clients)} карт, известный возраст у {clients.age.notna().sum()}, медиана {median_age:g} лет. Размеры всех возрастных и городских групп — в таблицах выше."))

Медиана возраста: 39.0
{'45–59': 85, '25–34': 61, '35–44': 60, '14–24': 38, '60–100': 8}
{'Пермь': 90, 'Казань': 88, 'Тюмень': 46, 'Уфа': 24, 'Екатеринбург': 22, 'Москва': 8}


**Текущий портрет (clients_selected.csv):** 294 карт, известный возраст у 252, медиана 39 лет. Размеры всех возрастных и городских групп — в таблицах выше.

**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** Медианный возраст 39 лет по 251 анкете. В Перми 89, Казани 88 анкет; в группе 60–100 всего 8. Возраст рассчитан в полных годах, границы 14 и 100 включены. Размеры групп показаны рядом; города без известного значения в словарь не входят, их доля дана в качестве.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 4.2. Активность и соединение

In [6]:
orders = orders.copy()
orders["client_id"] = orders.client_id.str.strip().str.upper()
stats = orders.groupby("client_id").agg(orders=("order_id","size"),revenue=("amount","sum"))
guests = clients.merge(stats,on="client_id",how="left",validate="one_to_one")
guests[["orders","revenue"]] = guests[["orders","revenue"]].fillna(0)
active_pct = round((guests.orders>0).mean()*100,1)
n_orphans = len(set(orders.client_id.dropna())-set(clients.client_id))
assert len(guests)==len(clients)
print('Активных, %:',active_pct,'; сирот с картой:',n_orphans)
print('Без карты, выручка:',orders.loc[orders.client_id.isna(),'amount'].sum())
print('Выручка гостей:',guests.revenue.sum())
assert guests.revenue.sum() + orders.loc[orders.client_id.isna(),'amount'].sum() == orders.amount.sum()
display(Markdown(f"**Текущая активность ({CLIENTS_FILE}):** {int((guests.orders>0).sum())} из {len(guests)} карт, {active_pct:.1f}%. Без покупок {int((guests.orders==0).sum())}; выручка таких карт равна 0. Сирот с картой {n_orphans}."))

Активных, %: 88.4 ; сирот с картой: 0
Без карты, выручка: 282245
Выручка гостей: 427950.0


**Текущая активность (clients_selected.csv):** 260 из 294 карт, 88.4%. Без покупок 34; выручка таких карт равна 0. Сирот с картой 0.

**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** Активны 88.7% из 293 гостей; сирот с картой 0. Выручка 427950 по картам плюс 282245 без карт составляет 710195. Гости без заказов остаются с нулевой выручкой. Проверка сирот отделена от заказов без карты. Равенство общей выручки защищает от незаметной потери строк.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 4.3. Откуда приходят гости

In [7]:
src = guests.source.fillna("не указано")
counts = src.value_counts()
guests["source"] = src.where(src.isin(counts[counts>=10].index),"прочее")
by_source = guests.assign(active=guests.orders>0).groupby("source").agg(clients=("client_id","size"),active_pct=("active","mean"),avg_revenue=("revenue","mean"))
by_source["active_pct"] = (by_source.active_pct*100).round(1)
by_source["avg_revenue"] = by_source.avg_revenue.round(0)
by_source = by_source.sort_values("clients",ascending=False)
best_source = by_source.loc[by_source.clients>=20,"avg_revenue"].idxmax()
print(by_source)
print('Лучший источник среди групп от 20:',best_source)
display(Markdown(f"**Текущие источники:** лучший среди групп от 20 карт — {best_source}; размер {int(by_source.loc[best_source,'clients'])}, средняя выручка {by_source.loc[best_source,'avg_revenue']:.0f} руб. на карту. Всего карт в сводке {int(by_source.clients.sum())}."))

            clients  active_pct  avg_revenue
source                                      
соцсети          80        87.5       1545.0
друзья           79        88.6       1413.0
мимо             41        87.8       1327.0
карты            32        87.5       1481.0
не указано       27        85.2       1523.0
реклама          27        92.6       1449.0
прочее            8       100.0       1340.0
Лучший источник среди групп от 20: соцсети


**Текущие источники:** лучший среди групп от 20 карт — соцсети; размер 80, средняя выручка 1545 руб. на карту. Всего карт в сводке 294.

**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** Соцсети: 80 анкет, 87.5% активных, 1545 руб. на гостя; друзья: 78 анкет, 1431 руб. Выручка на гостя включает неактивных. Группы меньше 20 не участвуют в выборе лучшего источника; это описательная связь, причинность и окупаемость рекламы из неё не следуют.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 4.4. Возраст и деньги

In [8]:
by_age = guests[guests.orders>0].groupby("age_group",observed=True).agg(guests_n=("client_id","size"),avg_orders=("orders","mean"),avg_revenue=("revenue","mean")).round(1)
best_group = by_age.avg_revenue.idxmax()
print(by_age)
print('Лидер:',best_group)
display(Markdown(f"**Текущие возрастные покупки:** лидер {best_group}, {int(by_age.loc[best_group,'guests_n'])} активных карт, средняя выручка {by_age.loc[best_group,'avg_revenue']:.1f} руб. Это сравнение активных гостей с известной датой рождения."))

           guests_n  avg_orders  avg_revenue
age_group                                   
14–24            35         5.2       1744.6
25–34            54         4.5       1477.0
35–44            57         4.6       1701.3
45–59            70         4.9       1655.7
60–100            8         5.0       1681.9
Лидер: 14–24


**Текущие возрастные покупки:** лидер 14–24, 35 активных карт, средняя выручка 1744.6 руб. Это сравнение активных гостей с известной датой рождения.

**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** Лидер 14–24: 35 активных гостей, 1744.6 руб. на гостя; 60–100: лишь 8 гостей, 1681.9 руб. Это сравнение активных гостей с известным возрастом. Размер группы ограничивает уверенность; пропуски даты могут смещать картину и требуют отдельного исследования.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 4.5. Свой город и поездки

In [9]:
shops = pd.read_csv(DATA / "shops.csv")
t = orders.dropna(subset=["client_id"]).merge(shops[["shop_id","city"]],on="shop_id",how="left",validate="many_to_one").rename(columns={"city":"shop_city"}).merge(clients[["client_id","city"]],on="client_id",how="left",validate="many_to_one")
known = t.dropna(subset=["shop_city","city"])
home_pct = round((known.shop_city==known.city).mean()*100,1)
travelers = (t.groupby("client_id").shop_city.nunique()>=2).sum()
print('Заказов в своём городе, %:',home_pct,'; путешественников:',travelers)
display(Markdown(f"**Текущая география:** {home_pct:.1f}% заказов с известными двумя городами сделаны в своём городе; в нескольких городах покупали {travelers} карт."))

Заказов в своём городе, %: 85.1 ; путешественников: 73


**Текущая география:** 85.1% заказов с известными двумя городами сделаны в своём городе; в нескольких городах покупали 73 карт.

**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** В своём городе 85.1% заказов с двумя известными городами; в двух и более городах покупали 73 гостя. Доля рассчитана только при двух известных городах. Число городов заказов не доказывает переезд; можно проверить кросс-городские кампании.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 4.6. Повторяемость

In [10]:
def report(raw,orders):
    clients = clean_clients(raw)
    clients["age"] = 2026-clients.birth_date.dt.year-(clients.birth_date.dt.strftime("%m%d")>"1201")
    clients["age_group"] = pd.cut(clients.age,bins=[14,25,35,45,60,101],labels=["14–24","25–34","35–44","45–59","60–100"],right=False)
    orders = orders.copy()
    orders["client_id"] = orders.client_id.str.strip().str.upper()
    stats = orders.groupby("client_id").agg(orders=("order_id","size"),revenue=("amount","sum"))
    guests = clients.merge(stats,on="client_id",how="left",validate="one_to_one")
    guests[["orders","revenue"]] = guests[["orders","revenue"]].fillna(0)
    return {"clients":len(clients),"active_pct":round((guests.orders>0).mean()*100,1),"median_age":clients.age.median(),"avg_revenue_active":round(guests.loc[guests.orders>0,"revenue"].mean(),1)}
summary = report(raw,orders)
nov = report(raw,orders[orders.datetime.dt.month==11])
alternate = PRIVATE / 'clients_copy.csv'
shutil.copy(DATA / CLIENTS_FILE,alternate)
assert report(pd.read_csv(alternate),orders)==summary
print('Вся осень:',summary)
print('Ноябрь:',nov)
display(Markdown(f"**Итог выбранной выгрузки {CLIENTS_FILE}:** {summary['clients']} карт, активны {summary['active_pct']:.1f}%, медиана возраста {summary['median_age']:g}; выручка активного гостя {summary['avg_revenue_active']:.1f} руб. Журнал входа: {len(raw)} → {len(raw.drop_duplicates())} → {len(clients)}."))

Вся осень: {'clients': 294, 'active_pct': np.float64(88.4), 'median_age': np.float64(39.0), 'avg_revenue_active': np.float64(1646.0)}
Ноябрь: {'clients': 294, 'active_pct': np.float64(72.8), 'median_age': np.float64(39.0), 'avg_revenue_active': np.float64(705.5)}


**Итог выбранной выгрузки clients_selected.csv:** 294 карт, активны 88.4%, медиана возраста 39; выручка активного гостя 1646.0 руб. Журнал входа: 309 → 300 → 294.

In [11]:
sample = pd.to_datetime(pd.Series(['2012-11-30','1926-11-30']))
ref = pd.Timestamp('2026-11-29')
age_before = ref.year-sample.dt.year-((sample.dt.month>ref.month)|((sample.dt.month==ref.month)&(sample.dt.day>ref.day)))
ref = pd.Timestamp('2026-11-30')
age_day = ref.year-sample.dt.year-((sample.dt.month>ref.month)|((sample.dt.month==ref.month)&(sample.dt.day>ref.day)))
print('До дня рождения:',age_before.tolist(),'; в день:',age_day.tolist())
print('Неизвестный пол остаётся пропуском:',clean_gender(pd.Series(['море','желаю пропустить',None])).isna().all())
assert age_before.tolist()==[13,99] and age_day.tolist()==[14,100]
assert clean_gender(pd.Series(['море','желаю пропустить',None])).isna().all()

До дня рождения: [13, 99] ; в день: [14, 100]
Неизвестный пол остаётся пропуском: True


**Историческая сверка исходной выгрузки clients_raw.csv, 307 строк.** **Вывод.** За ноябрь активны 73.0% и выручка активного гостя 705.5 руб.; за осень 88.7% и 1646.0 руб. Функция использует переданные таблицы, ноябрь пересчитывается без переписывания очистки; копия анкет под другим именем даёт тот же отчёт.

Текущие числа выбранного CLIENTS_FILE приведены в динамических выводах расчётных ячеек.

## 5. Выводы и рекомендации
Актуальный числовой ответ выбранной выгрузки находится в динамическом итоге выше. Маркетингу следует сравнивать источники с затратами и размерами групп. Владельцу CRM — убрать заглушки и невозможные балансы, уточнить смысл дат. Программа лояльности не описывает покупателей без карты.

**Вывод.** Отчёт использует выбранный CLIENTS_FILE, а числовые подписи вычисляются при выполнении. Исторические сверки явно отделены.